<a href="https://colab.research.google.com/github/DhimanTarafdar/next-word-prediction-using-LSTM/blob/main/LSTM_project_code_explanation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Project — Complex Code Explanation Notes

---

## 1. Vocabulary Building — `Counter` দিয়ে Vocab তৈরি

```python
from collections import Counter

vocab = { '<unk>' : 0 }

for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)
```

**Explanation:**

`Counter(tokens)` হলো একটি Python built-in tool যেটা একটা list-এর প্রতিটা element কতবার আছে সেটা count করে। এখানে `.keys()` দিয়ে শুধু unique words গুলো বের করা হচ্ছে — frequency দরকার নেই, শুধু কোন কোন word আছে সেটা দরকার।

`vocab` হলো একটা **Dictionary** যেখানে প্রতিটা word-এর একটা unique **integer index** assign করা হচ্ছে।

- `'<unk>'` মানে **Unknown token** — যে word vocab-এ নেই সেটাকে এই index দেওয়া হয়।
- `len(vocab)` use করা হচ্ছে কারণ প্রতিবার নতুন word যোগ হলে vocab-এর size বাড়ে, তাই index automatically increment হয়।
**Example:**
```
tokens = ["the", "cat", "sat", "the"]
Counter(tokens) → {"the": 2, "cat": 1, "sat": 1}
Counter(tokens).keys() → ["the", "cat", "sat"]

vocab = {"<unk>": 0, "the": 1, "cat": 2, "sat": 3}
```

---

## 2. Text to Indices — `text_to_indices()` Function

```python
def text_to_indices(sentence, vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
    return numerical_sentence
```

**Explanation:**

Neural Network text বুঝতে পারে না, সে শুধু **number** বোঝে। তাই এই function প্রতিটা word-কে তার vocab-এ assigned **integer index**-এ convert করে।

- যদি word vocab-এ **থাকে** → সেই word-এর index নাও
- যদি word vocab-এ **না থাকে** → `<unk>`-এর index (0) দাও
**Example:**
```
vocab = {"<unk>": 0, "the": 1, "cat": 2, "sat": 3}
sentence = ["the", "dog", "sat"]

→ [1, 0, 3]
("dog" vocab-এ নেই, তাই <unk> = 0)
```

---

## 3. Training Sequence তৈরি — Sliding Window Logic

```python
training_sequence = []
for sentence in input_numerical_sentence:
    for i in range(1, len(sentence)):
        training_sequence.append(sentence[:i+1])
```

**Explanation:**

এটা LSTM-এর জন্য **next word prediction** training data তৈরির সবচেয়ে গুরুত্বপূর্ণ step।

ধরো sentence হলো: `[10, 25, 47, 62]`

Inner loop যা করে:
- `i=1` → `[10, 25]`
- `i=2` → `[10, 25, 47]`
- `i=3` → `[10, 25, 47, 62]`
এভাবে প্রতিটি sub-sequence তৈরি হয় যেখানে **শেষ word হলো target (y)** এবং তার আগের সব word হলো **input (X)**।

মানে model শেখে: "এই words গুলো দেখলে পরের word কী হবে?"

এটাকে বলা হয় **Sliding Window** বা **n-gram style training**।

---

## 4. Padding — Variable Length Sequences সমান করা

```python
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append(
        [0] * (max(len_list) - len(sequence)) + sequence
    )
```

**Explanation:**

LSTM-এ সব sequence-এর **length সমান** থাকতে হয় কারণ PyTorch tensor-এ rectangular matrix দরকার।

যেমন:
- max length = 25
- একটা sequence = `[10, 25, 47]` (length 3)
- Padding করলে → `[0, 0, 0, 0, ...(22 টা শূন্য)..., 10, 25, 47]`
**Left padding** করা হচ্ছে (শূন্যগুলো বাম দিকে বসছে) যাতে **শেষের actual content** সবসময় একই position-এ থাকে। `0` হলো padding index — এটাই `<unk>`-এর index — কিন্তু এখানে এটা শুধু placeholder হিসেবে ব্যবহার হচ্ছে।

---

## 5. X এবং y Split — Input ও Target আলাদা করা

```python
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:, -1]
```

**Explanation:**

এটা **NumPy/PyTorch slicing** দিয়ে করা হচ্ছে।

- `[:, :-1]` মানে: সব rows নাও, কিন্তু **শেষ column বাদ দাও** → এটাই **input X**
- `[:, -1]` মানে: সব rows-এর **শুধু শেষ column** নাও → এটাই **target y**
**Example:**
```
padded_sequence = [0, 0, 10, 25, 47]

X = [0, 0, 10, 25]   ← এই words দেখে
y = [47]              ← এই word predict করতে হবে
```

Model শেখে: X দেখে y predict করতে।

---

## 6. Custom Dataset Class — PyTorch `Dataset` Inherit করা

```python
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
```

**Explanation:**

PyTorch-এ নিজের data নিয়ে কাজ করতে হলে `torch.utils.data.Dataset` class-কে **inherit** করে একটা custom class বানাতে হয়।

তিনটা method থাকতেই হবে:

| Method | কাজ |
|---|---|
| `__init__` | Data store করো |
| `__len__` | Dataset-এ মোট কতটা sample আছে সেটা return করো |
| `__getitem__` | index দিলে সেই index-এর (X, y) pair return করো |

`__getitem__` দরকার কারণ `DataLoader` automatically random index দিয়ে data fetch করে। `self.X.shape[0]` মানে rows-এর সংখ্যা — মানে মোট sample count।

---

## 7. LSTM Model Architecture — সবচেয়ে Complex Part

```python
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.fc = nn.Linear(150, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
        output = self.fc(final_hidden_state.squeeze(0))
        return output
```

**Explanation:**

এটা সম্পূর্ণ LSTM model-এর architecture। তিনটা layer আছে:

**Layer 1 — `nn.Embedding(vocab_size, 100)`:**
- Integer index (word number) নিয়ে সেটাকে একটা **100-dimensional dense vector**-এ convert করে।
- কারণ: word index `[47]` meaningless, কিন্তু embedding vector-এ words-এর মধ্যে semantic relationship থাকে।
- Input shape: `(batch_size, sequence_length)` → Output shape: `(batch_size, sequence_length, 100)`
**Layer 2 — `nn.LSTM(100, 150, batch_first=True)`:**
- `input_size=100` → embedding vector-এর size নেয়
- `hidden_size=150` → LSTM-এর internal memory size
- `batch_first=True` → input-এর প্রথম dimension = batch (default হলো sequence first)
- Output দেয় দুটো জিনিস:
  - `intermediate_hidden_states` → প্রতি time step-এর hidden state (shape: `batch, seq_len, 150`)
  - `(final_hidden_state, final_cell_state)` → শেষ time step-এর hidden ও cell state
**Layer 3 — `nn.Linear(150, vocab_size)`:**
- LSTM-এর final hidden state নিয়ে সেটাকে vocab_size মাপের vector-এ project করে।
- এটাই **logits** — প্রতিটা word-এর score।
**`final_hidden_state.squeeze(0)` কেন?**
- `final_hidden_state`-এর shape হলো `(1, batch_size, 150)` — প্রথম `1` হলো LSTM layers-এর সংখ্যা।
- `.squeeze(0)` সেই extra dimension সরিয়ে দেয় → shape হয় `(batch_size, 150)`
- এটা না করলে `nn.Linear` shape mismatch error দেবে।
**Data flow চিত্র:**
```
Input X (integers)
     ↓
Embedding Layer → (batch, seq_len, 100)
     ↓
LSTM Layer → final_hidden_state → (1, batch, 150)
     ↓
squeeze(0) → (batch, 150)
     ↓
Linear Layer → (batch, vocab_size)
     ↓
Output (logits for each word)
```

---

## 8. Training Loop — সম্পূর্ণ Learning Process

```python
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        output = model(batch_x)

        loss = criterion(output, batch_y)

        loss.backward()

        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")
```

**Explanation:**

**`nn.CrossEntropyLoss()`:**
- Classification-এর জন্য সবচেয়ে common loss function।
- Model-এর output (logits) এবং actual word index (y) নিয়ে কতটা ভুল হয়েছে সেটা measure করে।
- Inside-এ automatically `Softmax` apply করে।
**`torch.optim.Adam`:**
- Gradient descent-এর advanced version।
- `lr=0.001` মানে প্রতি step-এ weights কতটুকু update হবে।
- `model.parameters()` দিলে Adam জানে কোন কোন weights update করতে হবে।
**Training loop-এর ৫টা গুরুত্বপূর্ণ step:**

| Step | Code | কারণ |
|---|---|---|
| 1. GPU-তে পাঠাও | `.to(device)` | GPU থাকলে faster computation |
| 2. Gradient মুছো | `zero_grad()` | আগের batch-এর gradient জমে না |
| 3. Forward pass | `model(batch_x)` | Prediction বের করো |
| 4. Loss calculate | `criterion(output, y)` | কতটা ভুল হলো |
| 5. Backward + Update | `backward()` + `step()` | Weights ঠিক করো |

**`optimizer.zero_grad()` কেন দরকার?**
PyTorch-এ gradient **accumulate** হয় — মানে আগের calculation-এর gradient পরেরটায় যোগ হতে থাকে। প্রতিটা batch-এর আগে সেটা মুছে দিতে হয় নাহলে training corrupt হয়।

---